In [1]:
import requests
import pyodbc
import logging
from datetime import datetime

In [2]:
API_KEY = "70790c7256597541bf1857fe1c0ee809"
My_DRIVER ="ODBC Driver 18 for SQL Server"
My_SERVER = r"DESKTOP-5SI8TJO\SQLEXPRESS;"
My_DATABASE = "t_sql_database;"

In [3]:
#Getting the Data:

url_1 = f"http://api.weatherstack.com/current?access_key={API_KEY}&query=Cairo"
User_1 = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
report = requests.get(url = url_1, headers = {"User-Agent":User_1})

In [4]:
data = report.json()

In [5]:
data

{'request': {'type': 'City',
  'query': 'Cairo, Egypt',
  'language': 'en',
  'unit': 'm'},
 'location': {'name': 'Cairo',
  'country': 'Egypt',
  'region': 'Al Qahirah',
  'lat': '30.050',
  'lon': '31.250',
  'timezone_id': 'Africa/Cairo',
  'localtime': '2026-08-28 18:05',
  'localtime_epoch': 1787940300,
  'utc_offset': '3.0'},
 'current': {'observation_time': '03:05 PM',
  'temperature': 38,
  'weather_code': 113,
  'weather_icons': ['https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0001_sunny.png'],
  'weather_descriptions': ['Sunny'],
  'astro': {'sunrise': '05:30 AM',
   'sunset': '06:22 PM',
   'moonrise': '06:32 PM',
   'moonset': '05:32 AM',
   'moon_phase': 'Full Moon',
   'moon_illumination': 100},
  'air_quality': {'co': '164',
   'no2': '9.5',
   'o3': '94',
   'so2': '20.2',
   'pm2_5': '13.9',
   'pm10': '29.8',
   'us-epa-index': '1',
   'gb-defra-index': '1'},
  'wind_speed': 18,
  'wind_degree': 290,
  'wind_dir': 'WNW',
  'pressure': 1005,
  'pre

In [6]:
#Connect to the database:
for driver in pyodbc.drivers():
    print(driver)

SQL Server
ODBC Driver 17 for SQL Server
ODBC Driver 18 for SQL Server
Microsoft Access Driver (*.mdb, *.accdb)
Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)
Microsoft Access Text Driver (*.txt, *.csv)
Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)


In [11]:
conn_str = (
    fr'DRIVER={{{My_DRIVER}}};'
    fr'SERVER={My_SERVER};'
    fr'DATABASE={My_DATABASE};'
    r'Trusted_Connection=yes;'
    r'Encrypt=no;'
)


In [12]:
conn_str

'DRIVER={ODBC Driver 18 for SQL Server};SERVER=DESKTOP-5SI8TJO\\SQLEXPRESS;;DATABASE=t_sql_database;;Trusted_Connection=yes;Encrypt=no;'

In [19]:
datetime.now()

datetime.datetime(2026, 8, 28, 18, 24, 12, 492811)

In [18]:
datetime.now()

datetime.datetime(2026, 8, 28, 18, 23, 52, 937721)

In [7]:
cnxn = pyodbc.connect(conn_str)
cursor = cnxn.cursor()

NameError: name 'cnxn' is not defined

## Data Conversion

In [20]:
city,country,region,temperature,date,observation_time= [data['location']['name'],data['location']['country'],data['location']['region'],
                                    data['current']['temperature'],data['location']['localtime'][:-6],data['location']['localtime'][-5:]]
                                    


In [21]:
objects_time = [data['current']['astro']['sunrise'],data['current']['astro']['sunset'],data['current']['astro']['moonrise']
                 ,data['current']['astro']['moonset']]

time_format = '%I:%M %p'
objects_time_formated = []
for i in objects_time:
    try:
        objects_time_formated.append(datetime.strptime(i,time_format))
    except ValueError as e:
        print(type(e))
        objects_time_formated.append('NULL')
    continue
str_format = '%H:%M'
objects_str_formated = []
for i in objects_time_formated:
    try:
        objects_str_formated.append(i.strftime(str_format))
    except AttributeError as e:
        objects_str_formated.append('NULL')
        print(type(e))
    continue

    

In [26]:
s= [1,2,3,4,5,6]
for i in s[:-1]:
    print(i)

1
2
3
4
5


In [22]:
objects_time

['05:30 AM', '06:22 PM', '06:32 PM', '05:32 AM']

In [23]:
objects_time_formated

[datetime.datetime(1900, 1, 1, 5, 30),
 datetime.datetime(1900, 1, 1, 18, 22),
 datetime.datetime(1900, 1, 1, 18, 32),
 datetime.datetime(1900, 1, 1, 5, 32)]

In [25]:
objects_str_formated

['05:30', '18:22', '18:32', '05:32']

In [37]:
cursor.execute("""INSERT INTO DAILY_EGYPT_WEATHER(CITY,COUNTRY,REGION,TEMPRATURE,DATE,[OBSERVATION TIME]) 
                   VALUES(?,?,?,?,?,?)""",
               [city,country,region,temperature,date,observation_time])

In [51]:
cursor.execute("""
UPDATE DAILY_EGYPT_WEATHER 
SET DESCRIPTION = CASE 
WHEN TEMPRATURE <= 10 THEN 'COLD' 
WHEN TEMPRATURE >10 AND TEMPRATURE <30 THEN 'GOOD' 
WHEN TEMPRATURE >30 AND TEMPRATURE <40 THEN 'HOT'
WHEN TEMPRATURE >40 THEN 'VERY HOT'
END
""")


In [77]:
cursor.execute("""
INSERT INTO DAILY_EGYPT_OBJECT_TIME(ID,SUNRISE,
SUNSET,
MOONRISE,
MOONSET) 
VALUES(@@IDENTITY,?,?,?,?)
"""
,objects_str_formated)

